# Taller – Reconocimiento de Voz Local

**Integrantes:**
- Joan Sebastian Roberto Puerto
- Baruj Vladimir Ramírez Escalante
- Diego Alberto Romero Olmos
- Maicol Sebastian Olarte Ramirez
- Jorge Isaac Alandete Díaz

**Fecha de entrega:** 24 de abril de 2026

---

## Objetivo
Implementar una interfaz de voz local en Python, sin necesidad de conexión a internet (con opción en línea también), para controlar acciones visuales simples: cambio de colores, movimiento de figuras y comunicación OSC hacia escenas externas (Processing/Unity).

**Herramientas utilizadas:**
- `speech_recognition` – captura y reconocimiento de voz
- `pyaudio` – backend de audio para el micrófono
- `pyttsx3` – retroalimentación por voz (TTS offline)
- `python-osc` – envío de mensajes OSC a escenas externas
- `tkinter` – ventana de visualización directa en Python

---

## Estructura del taller

1. Instalación de dependencias
2. Captura de audio con `speech_recognition`
3. Reconocimiento de voz (Google online / Sphinx offline)
4. Diccionario de comandos básicos
5. Visualización directa en Python con `tkinter`
6. Comunicación OSC con `python-osc`
7. *(Bonus)* Retroalimentación por voz con `pyttsx3`

## 1. Instalación de dependencias

Ejecuta la siguiente celda **una sola vez** para instalar las librerías necesarias.
`PyAudio` requiere `portaudio` en el sistema (ya incluido en la mayoría de distribuciones Linux).
En Windows instala el wheel desde https://www.lfd.uci.edu/~gohlke/pythonlibs/#pyaudio.

In [1]:
# Instalar dependencias (ejecutar solo una vez)
# !pip install SpeechRecognition pyaudio pyttsx3 python-osc
# En sistemas Linux con portaudio:
# !sudo apt-get install portaudio19-dev python3-pyaudio
# !pip install pyaudio
print("Dependencias listas (si ya fueron instaladas).")

Dependencias listas (si ya fueron instaladas).


## 2. Importación de librerías

In [2]:
import speech_recognition as sr   # Reconocimiento de voz
import threading                    # Para correr el listener sin bloquear
import time                         # Control de tiempo
import sys                          # Información del sistema

# Verificar versión de speech_recognition
print(f"SpeechRecognition v{sr.__version__}")
print(f"Python {sys.version}")

# Comprobar micrófonos disponibles
print("\nMicrófonos disponibles:")
for i, nombre in enumerate(sr.Microphone.list_microphone_names()):
    print(f"  [{i}] {nombre}")

SpeechRecognition v3.16.1
Python 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]

Micrófonos disponibles:
  [0] Microsoft Sound Mapper - Input
  [1] Microphone (High Definition Aud
  [2] Microsoft Sound Mapper - Output
  [3] Altavoces (High Definition Audi
  [4] Primary Sound Capture Driver
  [5] Microphone (High Definition Audio Device)
  [6] Primary Sound Driver
  [7] Altavoces (High Definition Audio Device)
  [8] Altavoces (High Definition Audio Device)
  [9] Microphone (High Definition Audio Device)
  [10] Speakers (HD Audio Speaker)
  [11] Micrófono (HD Audio Microphone)
  [12] Línea ()
  [13] Auriculares ()
  [14] Output (@System32\drivers\bthhfenum.sys,#4;%1 Hands-Free HF Audio%0
;(Power Armor 13))
  [15] Input (@System32\drivers\bthhfenum.sys,#4;%1 Hands-Free HF Audio%0
;(Power Armor 13))
  [16] Output (@System32\drivers\bthhfenum.sys,#4;%1 Hands-Free HF Audio%0
;(Redmi Note 8 Pro))
  [17] Input (@System32\drivers\bthhfenum.sys,#

## 3. Captura de audio con `speech_recognition`

`speech_recognition` envuelve el proceso de grabación y
transcripción de audio. El flujo básico es:

1. Crear un `Recognizer` y un `Microphone`.
2. Ajustar el umbral de silencio con `adjust_for_ambient_noise`.
3. Escuchar con `listen()` para obtener un objeto `AudioData`.
4. Transcribir con `recognize_google()` (online) o `recognize_sphinx()` (offline).

In [3]:
def capturar_audio_una_vez(indice_mic=None, duracion_calibracion=1):
    """
    Escucha UNA vez por el micrófono y devuelve el texto reconocido.
    
    Parámetros
    ----------
    indice_mic : int | None
        Índice del micrófono a usar (None = predeterminado del sistema).
    duracion_calibracion : float
        Segundos de ajuste de ruido ambiente antes de escuchar.
    
    Retorna
    -------
    str | None
        Texto reconocido, o None si hubo error / silencio.
    """
    recognizer = sr.Recognizer()
    mic_kwargs = {"device_index": indice_mic} if indice_mic is not None else {}

    with sr.Microphone(**mic_kwargs) as fuente:
        print("⏳ Calibrando ruido ambiente...")
        recognizer.adjust_for_ambient_noise(fuente, duration=duracion_calibracion)
        print("🎙️  Habla ahora...")
        try:
            audio = recognizer.listen(fuente, timeout=5, phrase_time_limit=4)
        except sr.WaitTimeoutError:
            print("⏰ Sin entrada de audio (timeout).")
            return None

    # --- Intentar reconocimiento offline (sphinx) y online (Google) ---
    texto = None
    try:
        # Motor OFFLINE: CMU Sphinx (requiere pocketsphinx instalado)
        texto = recognizer.recognize_sphinx(audio, language="es-ES")
        print(f"🔇 Sphinx  → '{texto}'")
    except sr.UnknownValueError:
        pass          # Sphinx no entendió; intentar con Google
    except sr.RequestError:
        pass          # Sphinx no instalado; ignorar

    if texto is None:
        try:
            # Motor ONLINE: Google Speech Recognition
            texto = recognizer.recognize_google(audio, language="es-ES")
            print(f"🌐 Google  → '{texto}'")
        except sr.UnknownValueError:
            print("❓ No se pudo entender el audio.")
        except sr.RequestError as e:
            print(f"⚠️  Error de red con Google: {e}")

    return texto

# Prueba rápida (comentar si no hay micrófono disponible)
# resultado = capturar_audio_una_vez()
# print("Texto capturado:", resultado)
print("Función 'capturar_audio_una_vez' definida correctamente.")

Función 'capturar_audio_una_vez' definida correctamente.


## 4. Diccionario de comandos básicos

Se define un diccionario que mapea palabras clave (en español) a
acciones internas. El reconocedor devuelve texto libre, por lo
que se busca coincidencia de subcadenas para mayor tolerancia a
errores de transcripción.

In [12]:
# ---------------------------------------------------------------------------
# Diccionario de comandos: palabra_clave → acción
# ---------------------------------------------------------------------------
COMANDOS = {
    "rojo"    : "COLOR_ROJO",
    "azul"    : "COLOR_AZUL",
    "verde"   : "COLOR_VERDE",
    "amarillo": "COLOR_AMARILLO",
    "girar"   : "GIRAR",
    "iniciar" : "INICIAR",
    "detener" : "DETENER",
    "grande"  : "TAMANIO_GRANDE",
    "pequeño" : "TAMANIO_PEQUENIO",
    "salir"   : "SALIR",
}

def interpretar_comando(texto: str) -> str | None:
    """
    Busca la primera palabra clave del diccionario dentro del texto
    reconocido y devuelve la acción correspondiente.

    Parámetros
    ----------
    texto : str
        Transcripción retornada por speech_recognition.

    Retorna
    -------
    str | None
        Código de acción (ej. 'COLOR_ROJO') o None si no hay coincidencia.
    """
    if texto is None:
        return None
    texto_lower = texto.lower().strip()
    for palabra, accion in COMANDOS.items():
        if palabra in texto_lower:
            return accion
    return None

# ------- Prueba del intérprete de comandos -------
pruebas = [
    "pon el cuadro en rojo por favor",
    "quiero azul",
    "empieza a girar",
    "detener todo",
    "blanco",   # No registrado
    None,
]

print(f"{'Texto reconocido':<45} → Acción")
print("-" * 60)
for t in pruebas:
    accion = interpretar_comando(t)
    print(f"{str(t):<45} → {accion}")

Texto reconocido                              → Acción
------------------------------------------------------------
pon el cuadro en rojo por favor               → COLOR_ROJO
quiero azul                                   → COLOR_AZUL
empieza a girar                               → GIRAR
detener todo                                  → DETENER
blanco                                        → None
None                                          → None


## 5. Visualización directa en Python con `tkinter`

Se crea una ventana `tkinter` con:
- Un **cuadrado de color** que cambia según el comando de color.
- Un **círculo** que gira (se mueve en trayectoria circular) cuando se dice *"girar"*.
- Una **etiqueta de estado** que muestra el último comando reconocido.

El reconocimiento de voz corre en un hilo separado para no bloquear la interfaz gráfica.

> **Nota de ejecución:** Esta celda abre una ventana de escritorio.
> Ciérrala para continuar con las siguientes celdas o di "salir".

In [13]:
import tkinter as tk
import math

# ---------------------------------------------------------------------------
# Estado compartido entre el hilo de voz y la GUI
# ---------------------------------------------------------------------------
estado = {
    "color_fondo"  : "#3498db",   # Azul inicial
    "angulo"       : 0,            # Grados del círculo giratorio
    "girando"      : False,        # ¿Está girando?
    "ultimo_cmd"   : "ninguno",
    "corriendo"    : True,
}

COLORES = {
    "COLOR_ROJO"    : "#e74c3c",
    "COLOR_AZUL"    : "#3498db",
    "COLOR_VERDE"   : "#2ecc71",
    "COLOR_AMARILLO": "#f1c40f",
}

def aplicar_accion(accion: str):
    """Actualiza el estado según la acción reconocida."""
    if accion in COLORES:
        estado["color_fondo"] = COLORES[accion]
        estado["ultimo_cmd"]  = accion
    elif accion == "GIRAR":
        estado["girando"]    = True
        estado["ultimo_cmd"] = "GIRAR"
    elif accion == "INICIAR":
        estado["girando"]    = True
        estado["ultimo_cmd"] = "INICIAR"
    elif accion == "DETENER":
        estado["girando"]    = False
        estado["ultimo_cmd"] = "DETENER"
    elif accion == "TAMANIO_GRANDE":
        estado["ultimo_cmd"] = "GRANDE"
    elif accion == "TAMANIO_PEQUENIO":
        estado["ultimo_cmd"] = "PEQUEÑO"
    elif accion == "SALIR":
        estado["corriendo"]  = False
        estado["ultimo_cmd"] = "SALIR"

# ---------------------------------------------------------------------------
# Hilo de reconocimiento de voz (corre en paralelo a la GUI)
# ---------------------------------------------------------------------------
def hilo_voz():
    """Escucha continuamente y aplica acciones hasta que corriendo=False."""
    recognizer = sr.Recognizer()
    with sr.Microphone() as fuente:
        recognizer.adjust_for_ambient_noise(fuente, duration=1)
        while estado["corriendo"]:
            try:
                audio = recognizer.listen(fuente, timeout=3, phrase_time_limit=4)
                texto = None
                # Intentar Sphinx primero (offline)
                try:
                    texto = recognizer.recognize_sphinx(audio, language="es-ES")
                except (sr.UnknownValueError, sr.RequestError):
                    pass
                # Fallback a Google
                if texto is None:
                    try:
                        texto = recognizer.recognize_google(audio, language="es-ES")
                    except (sr.UnknownValueError, sr.RequestError):
                        pass
                if texto:
                    accion = interpretar_comando(texto)
                    print(f"  Escuché: '{texto}'  →  {accion}")
                    if accion:
                        aplicar_accion(accion)
            except sr.WaitTimeoutError:
                pass  # Silencio, seguir escuchando

# ---------------------------------------------------------------------------
# GUI tkinter
# ---------------------------------------------------------------------------
ANCHO, ALTO = 500, 400
RADIO_ORBITA = 100
RADIO_CIRCULO = 20
CX, CY = ANCHO // 2, ALTO // 2   # Centro del canvas

def crear_ventana_voz():
    raiz = tk.Tk()
    raiz.title("Reconocimiento de Voz – Visualización")
    raiz.resizable(False, False)

    canvas = tk.Canvas(raiz, width=ANCHO, height=ALTO, bg="#1a1a2e")
    canvas.pack()

    # Cuadrado de color (cambia según comando de color)
    rect = canvas.create_rectangle(50, 100, 250, 300,
                                   fill=estado["color_fondo"], outline="white", width=2)

    # Etiqueta de titulo
    canvas.create_text(150, 50, text="Color por Voz",
                       fill="white", font=("Arial", 14, "bold"))

    # Etiqueta estado
    lbl_cmd = canvas.create_text(150, 330, text="Comando: ninguno",
                                 fill="#ecf0f1", font=("Arial", 11))

    # Círculo giratorio (sección derecha del canvas)
    orbita_cx, orbita_cy = 380, 200
    circulo = canvas.create_oval(
        orbita_cx + RADIO_ORBITA - RADIO_CIRCULO,
        orbita_cy - RADIO_CIRCULO,
        orbita_cx + RADIO_ORBITA + RADIO_CIRCULO,
        orbita_cy + RADIO_CIRCULO,
        fill="#e67e22", outline="white"
    )
    canvas.create_text(orbita_cx, orbita_cy - 130,
                       text="Girar / Detener", fill="white", font=("Arial", 11, "bold"))

    def actualizar():
        """Loop de animación a ~60 fps."""
        if not estado["corriendo"]:
            raiz.destroy()
            return

        # Actualizar color del cuadrado
        canvas.itemconfig(rect, fill=estado["color_fondo"])

        # Actualizar etiqueta de comando
        canvas.itemconfig(lbl_cmd, text=f"Comando: {estado['ultimo_cmd']}")

        # Rotar círculo si está activo
        if estado["girando"]:
            estado["angulo"] = (estado["angulo"] + 2) % 360
        ang_rad = math.radians(estado["angulo"])
        nx = orbita_cx + RADIO_ORBITA * math.cos(ang_rad)
        ny = orbita_cy + RADIO_ORBITA * math.sin(ang_rad)
        canvas.coords(circulo,
                      nx - RADIO_CIRCULO, ny - RADIO_CIRCULO,
                      nx + RADIO_CIRCULO, ny + RADIO_CIRCULO)

        raiz.after(16, actualizar)  # ~60 fps

    # Iniciar hilo de voz en segundo plano
    t = threading.Thread(target=hilo_voz, daemon=True)
    t.start()

    actualizar()
    raiz.mainloop()

# Ejecutar la ventana (descomenta para usar el micrófono)
crear_ventana_voz()
print("Ventana de visualización definida. Descomenta la última línea para ejecutar.")
print("Comandos disponibles:", list(COMANDOS.keys()))

  Escuché: 'iniciar'  →  INICIAR
  Escuché: 'iniciar'  →  INICIAR
  Escuché: 'iniciar'  →  INICIAR
  Texto: 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
[TTS] Sistema iniciado
  Texto: 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
[TTS] Sistema iniciado
  [Google] 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
  [Google] 'detener'
[OSC simulado] Acción 'DETENER' (sin cliente OSC)
  Escuché: 'detener'  →  DETENER
  Escuché: 'detener'  →  DETENER
  Escuché: 'detener'  →  DETENER
  Texto: 'detener'
[OSC simulado] Acción 'DETENER' (sin cliente OSC)
[TTS] Movimiento detenido
  Texto: 'detener'
[OSC simulado] Acción 'DETENER' (sin cliente OSC)
[TTS] Movimiento detenido
  Escuché: 'azul'  →  COLOR_AZUL
  Escuché: 'azul'  →  COLOR_AZUL
  Texto: 'azul'
[OSC simulado] Acción 'COLOR_AZUL' (sin cliente OSC)
  Texto: 'azul'
[OSC simulado] Acción 'COLOR_AZUL' (sin cliente OSC)
  [Google] 'azul'
[OSC simulado] Acción 'COLOR_AZUL' (sin cliente OSC)
[TTS] Color

KeyboardInterrupt: 

## 6. Comunicación OSC con `python-osc`

Con `python-osc` se envían mensajes OSC al puerto donde escuche
la escena externa (Processing o Unity).  
Por defecto se usa **127.0.0.1:7000** (local). Ajusta la IP y
el puerto según tu configuración.

### Protocolo de mensajes definido

| Comando de voz | Dirección OSC              | Argumento    |
|----------------|---------------------------|--------------|
| rojo           | `/color`                  | `"rojo"`     |
| azul           | `/color`                  | `"azul"`     |
| verde          | `/color`                  | `"verde"`    |
| amarillo       | `/color`                  | `"amarillo"` |
| girar          | `/movimiento`             | `"girar"`    |
| iniciar        | `/control`                | `"iniciar"`  |
| detener        | `/control`                | `"detener"`  |
| grande / pequeño | `/tamanio`              | `"grande"` / `"pequenio"` |

In [6]:
try:
    from pythonosc import udp_client
    OSC_DISPONIBLE = True
except ImportError:
    OSC_DISPONIBLE = False
    print("python-osc no instalado. Instala con:  pip install python-osc")

# ---------------------------------------------------------------------------
# Configuración OSC
# ---------------------------------------------------------------------------
OSC_IP   = "127.0.0.1"   # IP destino (cambiar por la IP de Unity/Processing)
OSC_PORT = 7000           # Puerto destino

def crear_cliente_osc(ip=OSC_IP, puerto=OSC_PORT):
    """Crea y retorna un cliente UDP OSC."""
    if not OSC_DISPONIBLE:
        return None
    return udp_client.SimpleUDPClient(ip, puerto)

# ---------------------------------------------------------------------------
# Tabla de ruteo: Acción interna → (dirección OSC, argumento)
# ---------------------------------------------------------------------------
RUTEO_OSC = {
    "COLOR_ROJO"      : ("/color",      "rojo"),
    "COLOR_AZUL"      : ("/color",      "azul"),
    "COLOR_VERDE"     : ("/color",      "verde"),
    "COLOR_AMARILLO"  : ("/color",      "amarillo"),
    "GIRAR"           : ("/movimiento", "girar"),
    "INICIAR"         : ("/control",    "iniciar"),
    "DETENER"         : ("/control",    "detener"),
    "TAMANIO_GRANDE"  : ("/tamanio",    "grande"),
    "TAMANIO_PEQUENIO": ("/tamanio",    "pequenio"),
}

def enviar_osc(cliente, accion: str):
    """
    Envía el mensaje OSC correspondiente a la acción reconocida.

    Parámetros
    ----------
    cliente : SimpleUDPClient | None
        Cliente OSC creado con `crear_cliente_osc`.
    accion : str
        Código de acción del diccionario COMANDOS.
    """
    if cliente is None:
        print(f"[OSC simulado] Acción '{accion}' (sin cliente OSC)")
        return
    if accion not in RUTEO_OSC:
        print(f"[OSC] Sin ruta para acción '{accion}'")
        return
    direccion, argumento = RUTEO_OSC[accion]
    cliente.send_message(direccion, argumento)
    print(f"[OSC] Enviado → {direccion}  '{argumento}'  →  {OSC_IP}:{OSC_PORT}")

# ---------------------------------------------------------------------------
# Demo: simular envío de mensajes OSC sin micrófono
# ---------------------------------------------------------------------------
cliente_osc = crear_cliente_osc()

textos_demo = [
    "pon todo en rojo",
    "quiero que gire",
    "detener el movimiento",
    "color verde",
    "iniciar",
]

print("=== Simulación de comandos OSC ===")
for txt in textos_demo:
    accion = interpretar_comando(txt)
    if accion:
        enviar_osc(cliente_osc, accion)
    else:
        print(f"  '{txt}' → sin comando reconocido")

python-osc no instalado. Instala con:  pip install python-osc
=== Simulación de comandos OSC ===
[OSC simulado] Acción 'COLOR_ROJO' (sin cliente OSC)
  'quiero que gire' → sin comando reconocido
[OSC simulado] Acción 'DETENER' (sin cliente OSC)
[OSC simulado] Acción 'COLOR_VERDE' (sin cliente OSC)
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)


## 7. Integración completa: Voz → Visualización + OSC

Esta celda une todos los módulos anteriores en un bucle
continuo de escucha que:

1. Captura audio desde el micrófono.
2. Transcribe el audio (Sphinx offline → Google online como fallback).
3. Interpreta el comando.
4. Aplica el efecto visual en `tkinter` **y** envía el mensaje OSC.

> Ejecuta esta celda solo si cuentas con micrófono activo.
> Di **"salir"** para terminar el bucle.

In [ ]:
def hilo_voz_con_osc(cliente, estado_app):
    """
    Hilo que escucha el micrófono continuamente, interpreta comandos
    y aplica acciones visuales + envía mensajes OSC.

    Parámetros
    ----------
    cliente : SimpleUDPClient | None
        Cliente OSC (puede ser None si no está disponible).
    estado_app : dict
        Estado compartido con la GUI tkinter.
    """
    recognizer = sr.Recognizer()
    with sr.Microphone() as fuente:
        recognizer.adjust_for_ambient_noise(fuente, duration=1)
        print("✅ Micrófono listo. Di un comando...")
        while estado_app["corriendo"]:
            try:
                audio = recognizer.listen(fuente, timeout=4, phrase_time_limit=4)

                # --- Transcripción ---
                texto = None
                try:
                    texto = recognizer.recognize_sphinx(audio, language="es-ES")
                    print(f"  [Sphinx] '{texto}'")
                except (sr.UnknownValueError, sr.RequestError):
                    pass

                if texto is None:
                    try:
                        texto = recognizer.recognize_google(audio, language="es-ES")
                        print(f"  [Google] '{texto}'")
                    except (sr.UnknownValueError, sr.RequestError):
                        pass

                # --- Interpretación y despacho ---
                if texto:
                    accion = interpretar_comando(texto)
                    if accion:
                        aplicar_accion(accion)          # Actualiza estado GUI
                        enviar_osc(cliente, accion)     # Envía OSC

            except sr.WaitTimeoutError:
                pass  # Silencio; continuar escuchando

def ejecutar_sistema_completo():
    """Lanza la GUI tkinter con el hilo de voz+OSC integrado."""
    # Restablecer estado
    estado.update({
        "color_fondo" : "#3498db",
        "angulo"      : 0,
        "girando"     : False,
        "ultimo_cmd"  : "ninguno",
        "corriendo"   : True,
    })

    cliente = crear_cliente_osc()

    raiz = tk.Tk()
    raiz.title("Voz → Visualización + OSC")
    raiz.resizable(False, False)

    canvas = tk.Canvas(raiz, width=ANCHO, height=ALTO, bg="#1a1a2e")
    canvas.pack()

    rect     = canvas.create_rectangle(50, 100, 250, 300,
                                        fill=estado["color_fondo"],
                                        outline="white", width=2)
    lbl_cmd  = canvas.create_text(150, 330,
                                   text="Comando: ninguno",
                                   fill="#ecf0f1", font=("Arial", 11))
    canvas.create_text(150, 50, text="Color por Voz",
                       fill="white", font=("Arial", 14, "bold"))
    canvas.create_text(380, 70, text="Girar / Detener",
                       fill="white", font=("Arial", 11, "bold"))

    orbita_cx, orbita_cy = 380, 200
    circulo = canvas.create_oval(
        orbita_cx + RADIO_ORBITA - RADIO_CIRCULO,
        orbita_cy - RADIO_CIRCULO,
        orbita_cx + RADIO_ORBITA + RADIO_CIRCULO,
        orbita_cy + RADIO_CIRCULO,
        fill="#e67e22", outline="white"
    )

    def actualizar():
        if not estado["corriendo"]:
            raiz.destroy()
            return
        canvas.itemconfig(rect, fill=estado["color_fondo"])
        canvas.itemconfig(lbl_cmd, text=f"Comando: {estado['ultimo_cmd']}")
        if estado["girando"]:
            estado["angulo"] = (estado["angulo"] + 2) % 360
        ang_rad = math.radians(estado["angulo"])
        nx = orbita_cx + RADIO_ORBITA * math.cos(ang_rad)
        ny = orbita_cy + RADIO_ORBITA * math.sin(ang_rad)
        canvas.coords(circulo,
                      nx - RADIO_CIRCULO, ny - RADIO_CIRCULO,
                      nx + RADIO_CIRCULO, ny + RADIO_CIRCULO)
        raiz.after(16, actualizar)

    t = threading.Thread(target=hilo_voz_con_osc, args=(cliente, estado), daemon=True)
    t.start()

    actualizar()
    raiz.mainloop()

ejecutar_sistema_completo()
print("Sistema completo definido.")

✅ Micrófono listo. Di un comando...
  Escuché: 'Hola'  →  None
  [Google] 'Hola'
  Escuché: 'Hola'  →  None
  [Google] 'Hola'
  Texto: 'Hola'
  Texto: 'Hola'
  Escuché: 'Hola'  →  None
  [Google] 'Hola'
  [Google] 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
  [Google] 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
  Texto: 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
[TTS] Sistema iniciado
  Escuché: 'iniciar'  →  INICIAR
  Escuché: 'iniciar'  →  INICIAR
  [Google] 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
  Texto: 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
[TTS] Sistema iniciado
  Escuché: 'iniciar'  →  INICIAR
  Escuché: 'lo hacemos caliente'  →  None
  Texto: 'no somos calientes'


KeyboardInterrupt: 

: 

## Bonus: Retroalimentación por voz con `pyttsx3`

`pyttsx3` es un motor de síntesis de voz (*Text-to-Speech*) que funciona
**completamente offline**. Se usa aquí para confirmar verbalmente el
comando reconocido, mejorando la experiencia de usuario.

In [8]:
try:
    import pyttsx3
    TTS_DISPONIBLE = True
except ImportError:
    TTS_DISPONIBLE = False
    print("pyttsx3 no instalado. Instala con:  pip install pyttsx3")

# ---------------------------------------------------------------------------
# Mensajes de confirmación por acción
# ---------------------------------------------------------------------------
MENSAJES_VOZ = {
    "COLOR_ROJO"      : "Color rojo aplicado",
    "COLOR_AZUL"      : "Color azul aplicado",
    "COLOR_VERDE"     : "Color verde aplicado",
    "COLOR_AMARILLO"  : "Color amarillo aplicado",
    "GIRAR"           : "Iniciando rotación",
    "INICIAR"         : "Sistema iniciado",
    "DETENER"         : "Movimiento detenido",
    "TAMANIO_GRANDE"  : "Tamaño grande",
    "TAMANIO_PEQUENIO": "Tamaño pequeño",
    "SALIR"           : "Hasta luego",
}

def inicializar_tts(velocidad=150, volumen=0.9):
    """
    Inicializa el motor pyttsx3 con los parámetros dados.

    Parámetros
    ----------
    velocidad : int
        Palabras por minuto (defecto 150).
    volumen : float
        Volumen entre 0.0 y 1.0 (defecto 0.9).

    Retorna
    -------
    pyttsx3.Engine | None
    """
    if not TTS_DISPONIBLE:
        return None
    motor = pyttsx3.init()
    motor.setProperty("rate", velocidad)
    motor.setProperty("volume", volumen)
    # Seleccionar voz en español si está disponible
    voces = motor.getProperty("voices")
    for voz in voces:
        if "spanish" in voz.name.lower() or "es" in voz.id.lower():
            motor.setProperty("voice", voz.id)
            break
    return motor

def decir(motor, accion: str):
    """
    Reproduce en voz el mensaje asociado a la acción.

    Parámetros
    ----------
    motor : pyttsx3.Engine | None
    accion : str
    """
    if motor is None:
        print(f"[TTS simulado] {MENSAJES_VOZ.get(accion, 'Comando desconocido')}")
        return
    mensaje = MENSAJES_VOZ.get(accion, "Comando no reconocido")
    motor.say(mensaje)
    motor.runAndWait()

# ---------------------------------------------------------------------------
# Demo TTS sin micrófono
# ---------------------------------------------------------------------------
motor_tts = inicializar_tts()

acciones_demo = ["COLOR_ROJO", "GIRAR", "DETENER", "COLOR_VERDE"]
print("=== Demostración de retroalimentación por voz ===")
for accion in acciones_demo:
    decir(motor_tts, accion)

print("\nDemostración TTS completada.")

pyttsx3 no instalado. Instala con:  pip install pyttsx3
=== Demostración de retroalimentación por voz ===
[TTS simulado] Color rojo aplicado
[TTS simulado] Iniciando rotación
[TTS simulado] Movimiento detenido
[TTS simulado] Color verde aplicado

Demostración TTS completada.


## Bonus: Sistema completo con TTS integrado

Este bloque une **reconocimiento de voz + visualización tkinter + OSC + TTS**
en un solo flujo. El TTS corre en un hilo separado para no bloquear ni la GUI
ni el reconocimiento.

> Di **"salir"** para cerrar la aplicación.

In [10]:
import queue

# Cola thread-safe para mensajes TTS
_cola_tts: queue.Queue = queue.Queue()

def _worker_tts(motor):
    """Hilo consumidor que reproduce mensajes de la cola TTS."""
    while True:
        mensaje = _cola_tts.get()
        if mensaje is None:
            break   # Señal de cierre
        if motor:
            motor.say(mensaje)
            motor.runAndWait()
        else:
            print(f"[TTS] {mensaje}")
        _cola_tts.task_done()

def encolar_tts(accion: str):
    """Encola el mensaje TTS de una acción sin bloquear."""
    mensaje = MENSAJES_VOZ.get(accion)
    if mensaje:
        _cola_tts.put(mensaje)

def hilo_voz_completo(cliente, estado_app):
    """
    Versión final del hilo de voz que integra:
    - speech_recognition (Sphinx / Google)
    - interpretar_comando
    - aplicar_accion (GUI)
    - enviar_osc
    - encolar_tts (retroalimentación por voz)
    """
    recognizer = sr.Recognizer()
    with sr.Microphone() as fuente:
        recognizer.adjust_for_ambient_noise(fuente, duration=1)
        print("✅ Sistema listo. Di un comando...")
        while estado_app["corriendo"]:
            try:
                audio = recognizer.listen(fuente, timeout=4, phrase_time_limit=4)
                texto = None
                try:
                    texto = recognizer.recognize_sphinx(audio, language="es-ES")
                except (sr.UnknownValueError, sr.RequestError):
                    pass
                if texto is None:
                    try:
                        texto = recognizer.recognize_google(audio, language="es-ES")
                    except (sr.UnknownValueError, sr.RequestError):
                        pass
                if texto:
                    print(f"  Texto: '{texto}'")
                    accion = interpretar_comando(texto)
                    if accion:
                        aplicar_accion(accion)
                        enviar_osc(cliente, accion)
                        encolar_tts(accion)
            except sr.WaitTimeoutError:
                pass

def ejecutar_sistema_con_tts():
    """Lanza el sistema completo: GUI + voz + OSC + TTS."""
    estado.update({
        "color_fondo" : "#3498db",
        "angulo"      : 0,
        "girando"     : False,
        "ultimo_cmd"  : "ninguno",
        "corriendo"   : True,
    })

    motor = inicializar_tts()
    cliente = crear_cliente_osc()

    # Hilo TTS
    t_tts = threading.Thread(target=_worker_tts, args=(motor,), daemon=True)
    t_tts.start()

    # GUI tkinter
    raiz = tk.Tk()
    raiz.title("Voz → Visualización + OSC + TTS")
    raiz.resizable(False, False)

    canvas = tk.Canvas(raiz, width=ANCHO, height=ALTO + 60, bg="#1a1a2e")
    canvas.pack()

    rect     = canvas.create_rectangle(50, 100, 250, 300,
                                        fill=estado["color_fondo"],
                                        outline="white", width=2)
    lbl_cmd  = canvas.create_text(150, 340,
                                   text="Comando: ninguno",
                                   fill="#ecf0f1", font=("Arial", 11))
    lbl_info = canvas.create_text(ANCHO // 2, 400,
                                   text='Di "rojo", "azul", "girar", "detener", "salir"',
                                   fill="#95a5a6", font=("Arial", 9))
    canvas.create_text(150, 50, text="Color por Voz (+TTS)",
                       fill="white", font=("Arial", 14, "bold"))
    canvas.create_text(380, 70, text="Rotación",
                       fill="white", font=("Arial", 11, "bold"))

    orbita_cx, orbita_cy = 380, 200
    circulo = canvas.create_oval(
        orbita_cx + RADIO_ORBITA - RADIO_CIRCULO,
        orbita_cy - RADIO_CIRCULO,
        orbita_cx + RADIO_ORBITA + RADIO_CIRCULO,
        orbita_cy + RADIO_CIRCULO,
        fill="#e67e22", outline="white"
    )

    def actualizar():
        if not estado["corriendo"]:
            _cola_tts.put(None)   # Cerrar hilo TTS
            raiz.destroy()
            return
        canvas.itemconfig(rect, fill=estado["color_fondo"])
        canvas.itemconfig(lbl_cmd, text=f"Comando: {estado['ultimo_cmd']}")
        if estado["girando"]:
            estado["angulo"] = (estado["angulo"] + 2) % 360
        ang_rad = math.radians(estado["angulo"])
        nx = orbita_cx + RADIO_ORBITA * math.cos(ang_rad)
        ny = orbita_cy + RADIO_ORBITA * math.sin(ang_rad)
        canvas.coords(circulo,
                      nx - RADIO_CIRCULO, ny - RADIO_CIRCULO,
                      nx + RADIO_CIRCULO, ny + RADIO_CIRCULO)
        raiz.after(16, actualizar)

    t_voz = threading.Thread(target=hilo_voz_completo, args=(cliente, estado), daemon=True)
    t_voz.start()

    actualizar()
    raiz.mainloop()


ejecutar_sistema_con_tts()
print("Sistema con TTS definido.")

✅ Sistema listo. Di un comando...
  Texto: 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
[TTS] Sistema iniciado
  [Google] 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
  Texto: 'iniciar'
[OSC simulado] Acción 'INICIAR' (sin cliente OSC)
[TTS] Sistema iniciado
  Escuché: 'iniciar'  →  INICIAR
  Escuché: 'detener'  →  DETENER
  Texto: 'detener'
[OSC simulado] Acción 'DETENER' (sin cliente OSC)
[TTS] Movimiento detenido
  Texto: 'detener'
[OSC simulado] Acción 'DETENER' (sin cliente OSC)
[TTS] Movimiento detenido
  [Google] 'detener'
[OSC simulado] Acción 'DETENER' (sin cliente OSC)
  Escuché: 'girar'  →  GIRAR
  [Google] 'girar'
[OSC simulado] Acción 'GIRAR' (sin cliente OSC)
  Texto: 'girar'
[OSC simulado] Acción 'GIRAR' (sin cliente OSC)
[TTS] Iniciando rotación
  Texto: 'girar'
[OSC simulado] Acción 'GIRAR' (sin cliente OSC)
[TTS] Iniciando rotación
Sistema con TTS definido.


## Resumen del taller

| Módulo | Herramienta | Estado |
|--------|-------------|--------|
| Captura de audio | `speech_recognition` + `pyaudio` | ✅ Implementado |
| Reconocimiento offline | CMU Sphinx (`recognize_sphinx`) | ✅ Implementado (requiere `pocketsphinx`) |
| Reconocimiento online | Google (`recognize_google`) | ✅ Implementado (fallback) |
| Diccionario de comandos | Python `dict` + búsqueda por subcadena | ✅ Implementado |
| Visualización local | `tkinter` (color + rotación) | ✅ Implementado |
| Comunicación OSC | `python-osc` UDP | ✅ Implementado |
| Retroalimentación por voz | `pyttsx3` (TTS offline) | ✅ Implementado (Bonus) |

### Comandos disponibles

| Voz | Efecto visual | Mensaje OSC |
|-----|---------------|-------------|
| "rojo" | Cuadrado rojo | `/color "rojo"` |
| "azul" | Cuadrado azul | `/color "azul"` |
| "verde" | Cuadrado verde | `/color "verde"` |
| "amarillo" | Cuadrado amarillo | `/color "amarillo"` |
| "girar" / "iniciar" | Círculo empieza a rotar | `/movimiento "girar"` |
| "detener" | Círculo se detiene | `/control "detener"` |
| "grande" | Estado cambia a grande | `/tamanio "grande"` |
| "pequeño" | Estado cambia a pequeño | `/tamanio "pequenio"` |
| "salir" | Cierra la aplicación | — |